<a href="https://colab.research.google.com/github/NicolasRodrigues07/Sprint_IA/blob/main/Chatbot_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GoodWe EV Chatbot -  LLaMA 3.2  -  WeChat

Projeto Sprint 1

In [ ]:
%pip install --quiet langchain langchain-community langchain-core pypdf
%pip install --quiet langchain-huggingface transformers accelerate bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata, files
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print('Login HuggingFace OK')

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Login HuggingFace OK


In [ ]:
# Carregando o LLaMA 3.2-1B-Instruct
# https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                max_new_tokens=512, temperature=0.3, do_sample=True)

llm = HuggingFacePipeline(pipeline=pipe)
print('Modelo carregado!')

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo carregado!


In [ ]:
# Embeddings - tranformando em números
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)
vector_store = InMemoryVectorStore(embeddings)
print('Embeddings prontos!')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings prontos!


In [ ]:
# Upload dos PDFs(estamos usando 3)


pdf_files = ['datasheet.pdf', 'manual.pdf', 'chargegrid.pdf']
print(f'PDFs: {pdf_files}')
print(f'PDFs recebidos: {pdf_files}')

PDFs: ['datasheet.pdf', 'manual.pdf', 'chargegrid.pdf']
PDFs recebidos: ['datasheet.pdf', 'manual.pdf', 'chargegrid.pdf']


In [ ]:
# Carregando e indexando
all_docs = []
for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    all_docs.extend(docs)
    print(f'{pdf}: {len(docs)} paginas')

# splitting - recorta o texto
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(all_docs)
print(f'Split into {len(all_splits)} sub-documents.')

# guardando no vector store
document_ids = vector_store.add_documents(documents=all_splits)
print(f'Indexados {len(document_ids)} chunks. Pronto!')

datasheet.pdf: 2 paginas
manual.pdf: 70 paginas
chargegrid.pdf: 3 paginas
Split into 128 sub-documents.
Indexados 128 chunks. Pronto!


In [ ]:
# RAG com StateGraph - "Estuda os pdfs e responde"

PROMPT_TEMPLATE = (
    'Voce e um assistente especializado em eletropostos e veiculos eletricos da GoodWe. '
    'Use apenas o contexto abaixo para responder. Se nao souber, diga NAO SEI. '
    'Responda em portugues e seja objetivo.\n\n'
    'Pergunta: {question}\n\n'
    'Contexto: {context}\n\n'
    'Resposta:'
)

class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state['question'])
    return {'context': retrieved_docs}

def generate(state: State):
    docs_content = '\n\n'.join(doc.page_content for doc in state['context'])
    prompt = PROMPT_TEMPLATE.format(question=state['question'], context=docs_content)
    response = llm.invoke(prompt)
    # extrai apenas a resposta gerada apos 'Resposta:'
    answer = response.split('Resposta:')[-1].strip()
    return {'answer': answer}

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, 'retrieve')
graph = graph_builder.compile()

print('RAG pronto!')

RAG pronto!


In [ ]:
# Teste
result = graph.invoke({'question': 'O que e o ChargeGrid Intelligence e qual problema ele resolve?'})

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["answer"]}')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: [Document(id='82b82b0a-aff8-409d-8bdd-3fd2ac7004ee', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sistemas de Recarga Inteligente e Redes do Futuro', 'source': 'chargegrid.pdf', 'total_pages': 3, 'page': 2, 'page_label': '3'}, page_content='Charge Grids Inteligentes e a IA converte frotas de VEs em ativos de flexibilidade sistêmica, viabilizando\nredes mais limpas, baratas, resilientes e verdadeiramente autossustentáveis.\nBase conceitual expandida a partir de diretrizes de inovação em Smart Grids (Indicium AI, 2026).\nDocumento técnico gerado para fins informativos e de planejamento estratégico. \n• \n• \n1. \n2. \nRelatório Técnico: Charge Grids Inteligentes 3'), Document(id='9042cf8a-101e-4193-9d9e-6e6ef0645410', metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sistemas de Recarga Inteligente e Redes do Futuro', 'source': 'chargegrid.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2'},

In [ ]:
# Chatbot WeChat
print('=== GoodWe EV Chatbot ===')
print('Digite sair para encerrar')

while True:
    pergunta = input('\nVoce: ').strip()
    if pergunta.lower() in ['sair', 'exit']:
        print('Ate logo!')
        break
    if not pergunta:
        continue
    result = graph.invoke({'question': pergunta})
    print(f'\nBot: {result["answer"]}')
    print(f'(baseado em {len(result["context"])} trechos dos PDFs)')

=== GoodWe EV Chatbot ===
Digite sair para encerrar

Voce: sair
Ate logo!
